# Histology Backbone Transfer: Fine-tune on Oral Photos

Load the backbone pretrained on histopathology (from `histology_pretrain.ipynb`) and compare three strategies on the binary oral photo task:
1. **head_only** — Frozen backbone, train classifier head only
2. **progressive_K5** — Head warm-up (5 ep) then unfreeze last 5 backbone blocks
3. **full_finetune** — All parameters trainable from epoch 1

Uses the same oral train/val data and frozen test sets as `freeze_strategy_comparison.ipynb`.

## 1. Configuration

In [9]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"
BACKBONE_CKPT = NOTEBOOK_DIR / "results" / "histology_pretrain" / "backbone_state_dict.pth"

DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
DATA_DIR_BINARY = os.path.expanduser("~/dev/oral_cavity_binary_dataset")
DATA_DIR_SMALL_BINARY = os.path.expanduser("~/dev/oral-images-dataset/Oral-Image-small-binary-dataset")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"histology_transfer_{timestamp}"
OUTPUT_DIR = NOTEBOOK_DIR / "results" / "histology_transfer" / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
TB_LOG_DIR = OUTPUT_DIR / "tb_logs"
TB_LOG_DIR.mkdir(exist_ok=True)

CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "num_epochs": 50,
    "patience": 12,
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    "augmentation": "medical",
    "cancer_weight_multiplier": 1.5,
    "label_smoothing": 0.05,
    "strategies": {
        "head_only": {
            "description": "Frozen backbone, train head only",
            "lr_head": 1e-3,
            "freeze": "all_backbone",
        },
        "progressive_K5": {
            "description": "Head warm-up (5 ep) then unfreeze last 5 blocks",
            "lr_head": 1e-3,
            "lr_backbone": 1e-4,
            "warmup_epochs": 5,
            "unfreeze_last_k": 5,
        },
        "full_finetune": {
            "description": "All parameters trainable from epoch 1",
            "lr": 1e-4,
        },
    },
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "run_name": run_name, "backbone_ckpt": str(BACKBONE_CKPT)}, f, indent=2)

if not BACKBONE_CKPT.exists():
    raise FileNotFoundError(f"Run histology_pretrain.ipynb first. Missing: {BACKBONE_CKPT}")

print(f"Backbone: {BACKBONE_CKPT}")
print(f"Output: {OUTPUT_DIR}")
for name, s in CONFIG["strategies"].items():
    print(f"  {name}: {s['description']}")

Backbone: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_transfer/histology_transfer_20260218_175926
  head_only: Frozen backbone, train head only
  progressive_K5: Head warm-up (5 ep) then unfreeze last 5 blocks
  full_finetune: All parameters trainable from epoch 1


## 2. Imports and device

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

sys.path.insert(0, str(NOTEBOOK_DIR))
from frozen_test_sets import build_and_save_frozen_sets_v2, get_all_train_val_for_training

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NUM_CLASSES = 2

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

Using MPS


## 3. Build frozen test sets and merge training data

In [11]:
T5, TB, TC, TM, meta = build_and_save_frozen_sets_v2(
    data_dir_5class=DATA_DIR_5CLASS,
    data_dir_binary=DATA_DIR_BINARY,
    data_dir_small_binary=DATA_DIR_SMALL_BINARY,
    out_dir=FROZEN_DIR,
)

def collect_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend([str(p) for p in folder.glob(ext)])
    return out

binary_train_paths, binary_train_labels, _ = get_all_train_val_for_training(FROZEN_DIR)
five_dir = Path(DATA_DIR_5CLASS)
five_cancer, five_non = [], []
for split in ["train", "val"]:
    for c in ["OCA", "OPMD"]:
        five_cancer.extend(collect_images(five_dir / split / c))
    for c in ["Benign", "Healthy"]:
        five_non.extend(collect_images(five_dir / split / c))

all_paths_set = set()
all_cancer, all_non = [], []
for path, label in zip(binary_train_paths, binary_train_labels):
    if path not in all_paths_set:
        all_paths_set.add(path)
        (all_cancer if label == 1 else all_non).append(path)
for p in five_cancer:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_cancer.append(p)
for p in five_non:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_non.append(p)

all_paths = all_cancer + all_non
all_labels = [1] * len(all_cancer) + [0] * len(all_non)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}")
print(f"T5: {len(T5)}, TB: {len(TB)}, TC: {len(TC)}, TM: {len(TM)}")

Building frozen test sets v2...
  5-class dataset: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
  Binary dataset: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
  Small binary dataset: /Users/alokdwivedi/dev/oral-images-dataset/Oral-Image-small-binary-dataset

Frozen test sets v2 saved to /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
  T5: 534 samples (5-class test)
  TB: 246 samples (binary dataset test)
  TC: 49 samples (small binary dataset test)
  TM: 779 samples (mixed, all sources)
  binary_train_val_for_training: 1388 paths
  small_binary_train_val_for_training: 274 paths
Train: 3825, Val: 675
T5: 534, TB: 246, TC: 49, TM: 779


## 4. Augmentation and dataloaders

In [12]:
def get_augmentation_pipeline(mode="medical", image_size=(224, 224)):
    if HAS_ALBUMENTATIONS and mode == "medical":
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.RandomRotate90(p=0.4),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
            A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=15, val_shift_limit=12, p=0.4),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None

def get_val_transform(image_size=(224, 224)):
    if HAS_ALBUMENTATIONS:
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None

basic_train = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
basic_val = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_aug = get_augmentation_pipeline(CONFIG["augmentation"], CONFIG["image_size"])
val_aug = get_val_transform(CONFIG["image_size"])

class BinaryDataset(Dataset):
    def __init__(self, paths, labels, albu_transform=None, torch_transform=None):
        self.paths, self.labels = paths, labels
        self.albu_transform, self.torch_transform = albu_transform, torch_transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx])
        if img.mode in ("P", "PA"):
            img = img.convert("RGBA").convert("RGB")
        else:
            img = img.convert("RGB")
        label = self.labels[idx]
        if self.albu_transform is not None:
            img = self.albu_transform(image=np.array(img))["image"]
        elif self.torch_transform is not None:
            img = self.torch_transform(img)
        return img, label

train_ds = BinaryDataset(train_paths, train_labels, albu_transform=train_aug) if train_aug else BinaryDataset(train_paths, train_labels, torch_transform=basic_train)
val_ds = BinaryDataset(val_paths, val_labels, albu_transform=val_aug) if val_aug else BinaryDataset(val_paths, val_labels, torch_transform=basic_val)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 120, Val batches: 22


/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/venv_pytorch/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 5. Model: MobileNetV3 with histology-pretrained backbone

In [13]:
class ConservativeMobileNetV3(nn.Module):
    """MobileNetV3-Large with freeze/unfreeze helpers. Backbone can be loaded from histology checkpoint."""

    def __init__(self, num_classes=2, dropout_rate=0.2, backbone_ckpt_path=None):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(
                weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
            )
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )
        if backbone_ckpt_path and Path(backbone_ckpt_path).exists():
            state = torch.load(backbone_ckpt_path, map_location="cpu")
            self.load_state_dict(state, strict=False)
            print(f"Loaded backbone from {backbone_ckpt_path}")

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x

    def freeze_backbone(self):
        for p in self.backbone.features.parameters():
            p.requires_grad = False

    def unfreeze_last_k_blocks(self, k):
        children = list(self.backbone.features.children())
        for layer in children[-k:]:
            for p in layer.parameters():
                p.requires_grad = True

    def get_param_groups(self, lr_head, lr_backbone=None):
        groups = [{"params": list(self.classifier.parameters()), "lr": lr_head}]
        backbone_trainable = [p for p in self.backbone.features.parameters() if p.requires_grad]
        if backbone_trainable and lr_backbone is not None:
            groups.append({"params": backbone_trainable, "lr": lr_backbone})
        return groups

    def trainable_param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_param_count(self):
        return sum(p.numel() for p in self.parameters())


model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=0.2, backbone_ckpt_path=BACKBONE_CKPT)
print(f"Trainable: {model.trainable_param_count():,} / {model.total_param_count():,}")

Loaded backbone from /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
Trainable: 3,218,482 / 3,218,482


## 6. Class weights and training helpers

In [14]:
classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)
if CONFIG.get("cancer_weight_multiplier", 1.0) != 1.0:
    class_weights = class_weights.clone()
    class_weights[1] *= CONFIG["cancer_weight_multiplier"]
print(f"Class weights: {class_weights.tolist()}")

def _make_criterion():
    return nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=CONFIG.get("label_smoothing", 0.0),
    )

def _train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device).long()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total += target.size(0)
        correct += out.argmax(1).eq(target).sum().item()
    return total_loss / len(train_loader), 100.0 * correct / total

def _validate(model, val_loader, criterion, device):
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    probs_list, targets_list = [], []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device).long()
            out = model(data)
            loss = criterion(out, target)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)
            total += target.size(0)
            correct += out.argmax(1).eq(target).sum().item()
            probs_list.extend(probs[:, 1].cpu().numpy())
            targets_list.extend(target.cpu().numpy())
    acc = 100.0 * correct / total
    auc = roc_auc_score(targets_list, probs_list)
    return acc, auc, val_loss / len(val_loader)

Class weights: [1.0321100950241089, 1.454741358757019]


## 7. Strategy: head_only

In [15]:
def train_head_only(train_loader, val_loader, device, tb_log_dir=None):
    strat = CONFIG["strategies"]["head_only"]
    num_epochs = CONFIG["num_epochs"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=0.2, backbone_ckpt_path=BACKBONE_CKPT).to(device)
    model.freeze_backbone()

    criterion = _make_criterion()
    optimizer = optim.Adam(model.classifier.parameters(), lr=strat["lr_head"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    for epoch in range(num_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr = optimizer.param_groups[0]["lr"]
        if writer:
            writer.add_scalar("loss/train", avg_loss, epoch)
            writer.add_scalar("acc/val", val_acc, epoch)
            writer.add_scalar("auc/val", val_auc, epoch)
        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d}/{num_epochs}  loss={avg_loss:.4f}  val_acc={val_acc:.2f}%  val_auc={val_auc:.4f}")
        if patience_ctr >= CONFIG["patience"]:
            print(f"  Early stop at epoch {epoch+1}")
            break
        scheduler.step()
    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc

print("Strategy: head_only")
model_ho, acc_ho, auc_ho = train_head_only(train_loader, val_loader, device, tb_log_dir=str(TB_LOG_DIR / "head_only"))
torch.save(model_ho.state_dict(), OUTPUT_DIR / "models" / "head_only.pth")
print(f"Best val_acc={acc_ho:.2f}%, val_auc={auc_ho:.4f}")

Strategy: head_only
Loaded backbone from /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
  Ep   1/50  loss=0.6008  val_acc=65.78%  val_auc=0.7567
  Ep   5/50  loss=0.5125  val_acc=69.63%  val_auc=0.7987
  Ep  10/50  loss=0.4799  val_acc=73.19%  val_auc=0.8340
  Ep  15/50  loss=0.4520  val_acc=74.52%  val_auc=0.8509
  Ep  20/50  loss=0.4260  val_acc=74.81%  val_auc=0.8492
  Ep  25/50  loss=0.4030  val_acc=76.30%  val_auc=0.8542
  Ep  30/50  loss=0.3760  val_acc=75.85%  val_auc=0.8503
  Early stop at epoch 31
Best val_acc=76.44%, val_auc=0.8572


## 8. Strategy: progressive_K5

In [16]:
def train_progressive_K5(train_loader, val_loader, device, tb_log_dir=None):
    strat = CONFIG["strategies"]["progressive_K5"]
    warmup_epochs = strat["warmup_epochs"]
    main_epochs = CONFIG["num_epochs"]
    k = strat["unfreeze_last_k"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=0.2, backbone_ckpt_path=BACKBONE_CKPT).to(device)
    model.freeze_backbone()
    criterion = _make_criterion()
    optimizer = optim.Adam(model.classifier.parameters(), lr=strat["lr_head"])

    for epoch in range(warmup_epochs):
        _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        if epoch == warmup_epochs - 1:
            print(f"  Warmup done: val_auc={val_auc:.4f}")

    model.unfreeze_last_k_blocks(k)
    param_groups = model.get_param_groups(strat["lr_head"], strat["lr_backbone"])
    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    for epoch in range(main_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr = optimizer.param_groups[0]["lr"]
        if writer:
            writer.add_scalar("loss/train", avg_loss, warmup_epochs + epoch)
            writer.add_scalar("acc/val", val_acc, warmup_epochs + epoch)
            writer.add_scalar("auc/val", val_auc, warmup_epochs + epoch)
        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Ep {warmup_epochs + epoch + 1:3d}  loss={avg_loss:.4f}  val_acc={val_acc:.2f}%  val_auc={val_auc:.4f}")
        if patience_ctr >= CONFIG["patience"]:
            print(f"  Early stop at epoch {warmup_epochs + epoch + 1}")
            break
        scheduler.step()
    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc

print("Strategy: progressive_K5")
model_p5, acc_p5, auc_p5 = train_progressive_K5(train_loader, val_loader, device, tb_log_dir=str(TB_LOG_DIR / "progressive_K5"))
torch.save(model_p5.state_dict(), OUTPUT_DIR / "models" / "progressive_K5.pth")
print(f"Best val_acc={acc_p5:.2f}%, val_auc={auc_p5:.4f}")

Strategy: progressive_K5
Loaded backbone from /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
  Warmup done: val_auc=0.8167
  Ep   6  loss=0.5056  val_acc=71.85%  val_auc=0.8507
  Ep  10  loss=0.3610  val_acc=78.07%  val_auc=0.8752
  Ep  15  loss=0.2565  val_acc=80.30%  val_auc=0.8886
  Ep  20  loss=0.2237  val_acc=79.41%  val_auc=0.8838
  Ep  25  loss=0.1942  val_acc=80.15%  val_auc=0.8838
  Early stop at epoch 27
Best val_acc=80.30%, val_auc=0.8886


## 9. Strategy: full_finetune

In [17]:
def train_full_finetune(train_loader, val_loader, device, tb_log_dir=None):
    strat = CONFIG["strategies"]["full_finetune"]
    num_epochs = CONFIG["num_epochs"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=0.2, backbone_ckpt_path=BACKBONE_CKPT).to(device)
    criterion = _make_criterion()
    optimizer = optim.Adam(model.parameters(), lr=strat["lr"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    for epoch in range(num_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr = optimizer.param_groups[0]["lr"]
        if writer:
            writer.add_scalar("loss/train", avg_loss, epoch)
            writer.add_scalar("acc/val", val_acc, epoch)
            writer.add_scalar("auc/val", val_auc, epoch)
        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d}/{num_epochs}  loss={avg_loss:.4f}  val_acc={val_acc:.2f}%  val_auc={val_auc:.4f}")
        if patience_ctr >= CONFIG["patience"]:
            print(f"  Early stop at epoch {epoch+1}")
            break
        scheduler.step()
    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc

print("Strategy: full_finetune")
model_ff, acc_ff, auc_ff = train_full_finetune(train_loader, val_loader, device, tb_log_dir=str(TB_LOG_DIR / "full_finetune"))
torch.save(model_ff.state_dict(), OUTPUT_DIR / "models" / "full_finetune.pth")
print(f"Best val_acc={acc_ff:.2f}%, val_auc={auc_ff:.4f}")

Strategy: full_finetune
Loaded backbone from /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_pretrain/backbone_state_dict.pth
  Ep   1/50  loss=0.5704  val_acc=64.00%  val_auc=0.8022
  Ep   5/50  loss=0.3577  val_acc=77.48%  val_auc=0.8744
  Ep  10/50  loss=0.2456  val_acc=77.93%  val_auc=0.8641
  Ep  15/50  loss=0.2004  val_acc=77.93%  val_auc=0.8802
  Early stop at epoch 15
Best val_acc=74.22%, val_auc=0.8803


## 10. Summary

In [ ]:
summary = {
    "head_only": {"val_acc": acc_ho, "val_auc": auc_ho},
    "progressive_K5": {"val_acc": acc_p5, "val_auc": auc_p5},
    "full_finetune": {"val_acc": acc_ff, "val_auc": auc_ff},
}
with open(OUTPUT_DIR / "strategy_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Strategy summary (histology-pretrained backbone)")
print("-" * 50)
for name, s in summary.items():
    print(f"  {name:<20}  val_acc={s['val_acc']:.2f}%  val_auc={s['val_auc']:.4f}")
print(f"\nResults saved to {OUTPUT_DIR}")